# 🧪 Sprint 2: Agent Evaluation Workbench

This workbench is for **Real-World AI Engineering**. We move beyond vibes and start measuring how our agent handles different scenarios, phases, and personas.

## Goals:
1. **A/B Test Prompts**: Compare how different phase instructions affect behavior.
2. **Scenario Simulation**: Run the agent against "Golden Scenarios" (e.g., Honeymooners, Budget Solo).
3. **State Integrity**: Verify the agent updates the Rich State (`TripShape`, `MentalModel`) correctly.

In [1]:
import os
import json
import sys
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv

# Add the api directory to path so we can import our actual code
sys.path.append(os.path.abspath("../services/api"))

from agent.models import VacationPlan, VacationPlanPatch
from agent.orchestrator import AgentOrchestrator
from agent.prompt import SystemPrompts

# Load environment variables
load_dotenv("../.env")
print("✅ Workbench Initialized")

✅ Workbench Initialized


## 1. Scenario Runner
We define personas to see if the agent adapts its funnel (Discovery vs. Proposal) correctly.

In [2]:
SCENARIOS = [
    {
        "name": "The Vague Dreamer",
        "initial_message": "I just need to get away. Somewhere beautiful."
    },
    {
        "name": "The Goal-Oriented Planner",
        "initial_message": "Planning a 10-day honeymoon in Italy from NYC in June. Budget is $8k."
    },
    {
        "name": "The Budget Backpacker",
        "initial_message": "Looking for the cheapest possible beach vacation for me and a friend."
    }
]

def run_scenario(scenario_idx: int, turns: int = 3):
    scenario = SCENARIOS[scenario_idx]
    print(f"=== Testing Scenario: {scenario['name']} ===")
    
    orchestrator = AgentOrchestrator()
    plan = VacationPlan()
    history = []
    
    current_input = scenario["initial_message"]
    
    for i in range(turns):
        print(f"\n[Turn {i+1}] 👤 User: {current_input}")
        history.append({"role": "user", "content": current_input})
        
        response, plan, new_msgs = orchestrator.run_turn(history, plan)
        history.extend(new_msgs)
        history.append({"role": "assistant", "content": response})
        
        print(f"🤖 Agent: {response}")
        print(f"📊 State: PHASE={plan.phase} | UNKNOWNS={plan.mental_model.unknowns}")
        
        # For simulation, we'll just ask for more detail on turn 2
        if i == 0:
            current_input = "I'm thinking of Europe, maybe?"
        else:
            current_input = "What do you suggest?"

run_scenario(1) # Run the Goal-Oriented Planner

=== Testing Scenario: The Goal-Oriented Planner ===

[Turn 1] 👤 User: Planning a 10-day honeymoon in Italy from NYC in June. Budget is $8k.
⚠️ Groq rejected generation but we'll try to parse it anyway...
Tool Call Detected: {"phase": "context", "vacation_purpose": "honeymoon", "trip_shape": {"origin": "NYC", "duration_days": 10, "travelers": 2, "pax_description": "couple"}, "mental_model": {"knowns": [" destination: Italy", "travel dates: June"], "unknowns": ["accommodation preferences", "activities and sights to see", "transportation options"], "sentiments": ["excited", "romantic"]}, "candidates": ["Italy"], "budget_range": "$8k", "notes": "", "status": "in_progress"}
Tool handling error: 1 validation error for VacationPlanPatch
candidates.0
  Input should be a valid dictionary or instance of Destination [type=model_type, input_value='Italy', input_type=str]
    For further information visit https://errors.pydantic.dev/2.8/v/model_type
🤖 Agent: I had trouble updating the plan: 1 valid

Traceback (most recent call last):
  File "/Users/alastair/Github/vacation-planner/services/api/agent/orchestrator.py", line 131, in run_turn
    patch = VacationPlanPatch(**args)
            ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/alastair/Github/vacation-planner/.venv/lib/python3.11/site-packages/pydantic/main.py", line 193, in __init__
    self.__pydantic_validator__.validate_python(data, self_instance=self)
pydantic_core._pydantic_core.ValidationError: 1 validation error for VacationPlanPatch
candidates.0
  Input should be a valid dictionary or instance of Destination [type=model_type, input_value='Italy', input_type=str]
    For further information visit https://errors.pydantic.dev/2.8/v/model_type


Tool Call Detected: {"budget_range":"$8k","candidates":[{"country":"Italy","name":"","rationale":"","region":"","vibe":""}],"mental_model":{"knowns":["10-day honeymoon","from NYC","in June"],"sentiments":["excited","romantic"],"unknowns":["specific destination in Europe","budget details","couple's preferences"]},"notes":"","phase":"context","status":"in_progress","trip_shape":{"duration_days":10,"origin":"NYC","pax_description":"","travelers":2},"vacation_purpose":"honeymoon"}
🤖 Agent: You've mentioned Europe and even specified Italy, which is a wonderful choice for a honeymoon. June is a great time to visit Italy, the weather is warm and sunny, and there are many beautiful places to explore.

Since you're planning a 10-day honeymoon, I'd love to know what kind of experience you're looking for. Are you interested in exploring cities like Rome, Florence, and Venice, or do you prefer a more relaxed, countryside vibe?
📊 State: PHASE=context | UNKNOWNS=['specific destination in Europe', 'b

## 2. A/B Prompt Testing
Compare how the agent behaves with different SHARED_GUIDELINES.

In [3]:
def test_prompt_variation(variant_guidelines: str, user_input: str):
    # Temporarily override guidelines
    original = SystemPrompts.SHARED_GUIDELINES
    SystemPrompts.SHARED_GUIDELINES = variant_guidelines
    
    orchestrator = AgentOrchestrator()
    plan = VacationPlan()
    response, plan, _ = orchestrator.run_turn([{"role": "user", "content": user_input}], plan)
    
    print(f"--- GUIDELINES VARIANT ---\n{response}\n")
    
    SystemPrompts.SHARED_GUIDELINES = original

# Compare 'Strict Interrogator' vs 'Friendly Guide'
test_prompt_variation(
    "1. Be very formal. 2. Ask 5 questions at once. 3. Don't use tools unless essential.", 
    "I want to go to Tokyo"
)

Tool Call Detected: {"candidates":[{"country":"Japan","name":"Tokyo","rationale":"user preference","region":"Japan","vibe":"urban"}],"mental_model":{"knowns":["destination: Tokyo"],"sentiments":["excitement"],"unknowns":["vacation purpose","origin","duration of stay"]},"phase":"context","trip_shape":{"origin":null},"vacation_purpose":""}
--- GUIDELINES VARIANT ---
To further assist you in planning your ideal vacation to Tokyo, I have a few questions to help clarify your preferences. 

1. What is the primary purpose of your trip to Tokyo: is it for a celebration, an escape, or simply a break?
2. Could you please provide us with your city or country of origin, as this information will help us in suggesting the most convenient travel arrangements?
3. How many days do you envision your trip to Tokyo lasting, and are you flexible with your travel dates?
4. Are you traveling solo or with companions, and if so, what are their ages and interests?
5. What activities or experiences are you most 